In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from useful_fns import *

%load_ext autoreload
%autoreload 2

In [ ]:
data_dir = '../data/'
fig_dir = '../figures/calc_p_extrema_segments/'

In [ ]:
ds_bad_times = pd.read_csv(data_dir+'ds_bad_times.csv',header=None)[0].values #timestamps to split lc into segments

In [ ]:
fname = 'kic-007904418'
time, flux = load_lc(fname)

In [ ]:
time_q, flux_q = split_light_curve(time,flux)

In [ ]:
plt.figure()
for i in range(len(time_q)):
    plt.scatter(time_q[i],flux_q[i],s=0.5)
plt.xlabel('Time [Days]')
plt.ylabel('Flux')
plt.show()

In [ ]:
idx = 10
plt.figure()
plt.scatter(time_q[idx],flux_q[idx],s=0.05)
plt.xlabel('Time [Days]')
plt.ylabel('Flux')
plt.title(f'Q{idx} raw flux')
plt.savefig(fig_dir+f'Q{idx}_raw.png',bbox_inches='tight')

In [ ]:
stable_times, stable_fluxes = segment_split(time_q[idx],flux_q[idx],ds_bad_times)

In [ ]:
plt.figure()
for i in range(len(stable_times)):
    plt.scatter(stable_times[i],stable_fluxes[i],s=0.05)
plt.xlabel('Time [Days]')
plt.ylabel('Flux')
plt.title(f'Q{idx} split into segments')
plt.savefig(fig_dir+f'Q{idx}_seg.png',bbox_inches='tight')

In [ ]:
stable_fluxes_smoothed = [smooth(flux) for flux in stable_fluxes]

In [ ]:
plt.figure()
for i in range(len(stable_times)):
    plt.scatter(stable_times[i],stable_fluxes_smoothed[i],s=0.05)
plt.xlabel('Time [Days]')
plt.ylabel('Flux')
plt.title(f'Q{idx} segments smoothed')
plt.savefig(fig_dir+f'Q{idx}_seg_smooth.png',bbox_inches='tight')

In [ ]:
# Point-to-point differences
plt.figure()
N_crossings = 0
for i in range(len(stable_times)):
    dflux = np.diff(stable_fluxes_smoothed[i])
    signs = np.sign(dflux.data)
    crossing_idx  = np.where(signs[:-1] != signs[1:])[0]
    N_crossings = N_crossings + len(crossing_idx)
    dflux_time = stable_times[i][1:]
    # Times corresponding to dflux values
    dflux_time = stable_times[i][1:]


    # Interpolate the zero-crossing time
    t1 = dflux_time[crossing_idx]
    t2 = dflux_time[crossing_idx + 1]
    y1 = dflux[crossing_idx]
    y2 = dflux[crossing_idx + 1]

    crossing_time = t1 - y1 * (t2 - t1) / (y2 - y1)

    # Approximate flux at the extrema
    extrema_flux = np.interp(crossing_time, time, flux)
    plt.plot(stable_times[i],stable_fluxes_smoothed[i])
    plt.scatter(crossing_time,extrema_flux,s=5,c='black')
plt.xlabel('Time [Days]')
plt.ylabel('Flux')
plt.title(f'Q{idx} crossings marked, N = {N_crossings}')
plt.savefig(fig_dir+f'Q{idx}_seg_smooth_crossings.png',bbox_inches='tight')


In [ ]:
N_crossings